# 0 Data Preprocessing

In this notebook, we transform the raw Electricity Load Diagrams dataset into a format suitable for Machine Learning. 
We will clean the data, engineer time-based features, reshape the dataset from "wide" to "long" format, and create lag/rolling window features to capture historical consumption patterns. Finally, we will export the processed dataset.

## 1. Imports
We import the necessary libraries for data manipulation and date calculations. `dateutil.easter` is used to dynamically calculate the dates of movable Portuguese holidays.

In [1]:
import pandas as pd
import numpy as np
import datetime
from dateutil import easter
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import sys
import os
sys.path.append(os.path.abspath(".."))

from src.tools import add_temporal_features, get_national_weather, clean_clients, load_raw_data

df = load_raw_data("../Datasets/Electricity Dataset.csv")

Loading data from: ../Datasets/Electricity Dataset.csv...
Data loaded successfully. Shape: (140256, 371)


## 2. Temporal Feature Engineering
We apply the temporal features *before* melting the dataset. This is a crucial performance optimization: calculating the day of the week on 140,000 rows takes a fraction of a second, whereas doing it on 50 million rows (after melting) would take significantly longer.

In [2]:
df = add_temporal_features(df)


# Show resulst
display(df)

,Date,MT_001,MT_002,MT_003,MT_004,MT_005,MT_006,MT_007,MT_008,MT_009,...,MT_366,MT_367,MT_368,MT_369,MT_370,Weekday,Hour,Month,Is_Weekend,Is_Holiday
0,2011-01-01 00:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,6,1,2,True,True
1,2011-01-01 00:30:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,6,1,2,True,True
2,2011-01-01 00:45:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,6,1,2,True,True
3,2011-01-01 01:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,6,2,2,True,True
4,2011-01-01 01:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,6,2,2,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140251,2014-12-31 23:00:00,2.538071,22.048364,1.737619,150.406504,85.365854,303.571429,11.305822,282.828283,68.181818,...,5.851375,697.102722,176.961603,651.026393,7621.621622,3,24,13,False,False
140252,2014-12-31 23:15:00,2.538071,21.337127,1.737619,166.666667,81.707317,324.404762,11.305822,252.525253,64.685315,...,9.947338,671.641791,168.614357,669.354839,6702.702703,3,24,13,False,False
140253,2014-12-31 23:30:00,2.538071,20.625889,1.737619,162.601626,82.926829,318.452381,10.175240,242.424242,61.188811,...,9.362200,670.763828,153.589316,670.087977,6864.864865,3,24,13,False,False
140254,2014-12-31 23:45:00,1.269036,21.337127,1.737619,166.666667,85.365854,285.714286,10.175240,225.589226,64.685315,...,4.095963,664.618086,146.911519,646.627566,6540.540541,3,24,13,False,False


## 3. National Weather
Since we don't know where each client is located, we built a national average temperature as a proxy signal.
Rather than a simple average of 4 cities, we weight each city by its population: Lisbon (547k), Porto (237k), Faro (64k), Évora (56k). 

From that weighted temperature we also derive two extra columns:

- HDH (Heating Degree Hours): how many degrees below 18°C it is. When this is high, people are heating their homes, so consumption goes up. 

- CDH (Cooling Degree Hours): how many degrees above 18°C it is. When this is high, people are running air conditioning, so consumption goes up. 

18°C is the standard threshold for Heating/Cooling Degree Days (HDD/CDD) in energy analytics

In [4]:
weather_df = get_national_weather(start_date="2011-01-01", end_date="2014-12-31")

# Show results
display(weather_df)

  Fetching weather for Lisbon...
  Fetching weather for Porto...
  Fetching weather for Faro...
  Fetching weather for Evora...


,Date,HDH,CDH
0,2011-01-01 00:00:00,6.488938,0.0
1,2011-01-01 01:00:00,6.647566,0.0
2,2011-01-01 02:00:00,6.661394,0.0
3,2011-01-01 03:00:00,6.641925,0.0
4,2011-01-01 04:00:00,6.755199,0.0
...,...,...,...
35059,2014-12-31 19:00:00,8.685951,0.0
35060,2014-12-31 20:00:00,9.505310,0.0
35061,2014-12-31 21:00:00,10.293141,0.0
35062,2014-12-31 22:00:00,10.786615,0.0


## 6. Handling Inactive Periods

To ensure our models learn from actual consumption behavior, we must remove periods of total inactivity:
1. **Leading Zeros (Pre-activation):** Many clients joined the grid after 2011. We identify the first active timestamp for each client and remove the preceding inactive period.
2. **Inactive Clients (Churn prevention):** We identify clients with near-zero consumption in the last month of the dataset and remove them entirely. Forecasting flat zeros for churned users would artificially distort error metrics like MAPE.

In [6]:
rows_before = len(df)
df = clean_clients(df)
rows_after = len(df)

print(f"Cleaning complete.")
print(f"Rows removed: {rows_before - rows_after} ({(1 - rows_after/rows_before)*100:.1f}%)")

Trimming leading zeros (finding actual start date for each client)...


KeyError: 'Consumption'

## 7. Grouped Lag and Rolling Features

To forecast future consumption, the model needs to know recent past consumption. We engineer several time-shifted features to capture short, daily, and weekly seasonality profiles:
- **Lags:** What the client consumed 1 step ago (`Lag_15min`), 4 steps ago (`Lag_1h`), 1 day ago (`Lag_24h`), and 1 week ago (`Lag_1week`).
- **Rolling Windows:** The smoothed average consumption over the last 4 hours (`Rolling_Mean_4h`).

In [ ]:
# This cell should take less than 30 seconds to run

# ---------------------------------------------------------
# Sort chronologically per client before shifting
# ---------------------------------------------------------
print("Sorting data for lag calculations...")
df_long = df_long.sort_values(by=['ClientID', 'Date']).reset_index(drop=True)

# ---------------------------------------------------------
# Contiguity check: all timestamps should be exactly 15 minutes apart
# ---------------------------------------------------------
time_diffs = df_long.groupby('ClientID', observed=True)['Date'].diff()
irregular_gaps = time_diffs[(time_diffs.notnull()) & (time_diffs != pd.Timedelta(minutes=15))]
if len(irregular_gaps) != 0:
    print(f"Warning: Found {len(irregular_gaps)} missing or irregular timestamp intervals!")
    display(df_long.loc[irregular_gaps.index])

# ---------------------------------------------------------
# Consumption Lags (grouped by client to prevent cross-client bleed)
# ---------------------------------------------------------
print("Calculating consumption lags...")
df_long['Lag_15min'] = df_long.groupby('ClientID', observed=True)['Consumption'].shift(1).astype(np.float32)
df_long['Lag_1h']    = df_long.groupby('ClientID', observed=True)['Consumption'].shift(4).astype(np.float32)
df_long['Lag_24h']   = df_long.groupby('ClientID', observed=True)['Consumption'].shift(96).astype(np.float32)
df_long['Lag_1week'] = df_long.groupby('ClientID', observed=True)['Consumption'].shift(672).astype(np.float32)

print("Calculating rolling windows...")
df_long['Rolling_Mean_4h'] = df_long.groupby('ClientID', observed=True)['Consumption'].transform(
    lambda x: x.rolling(window=16).mean()
).astype(np.float32)

# ---------------------------------------------------------
# Weather Lags (thermal inertia: yesterday's temperature still affects today's demand)
# ---------------------------------------------------------
print("Calculating weather lags (24h thermal inertia)...")
df_long['HDH_lag24h'] = df_long.groupby('ClientID', observed=True)['HDH'].shift(96).astype(np.float32)
df_long['CDH_lag24h'] = df_long.groupby('ClientID', observed=True)['CDH'].shift(96).astype(np.float32)

# ---------------------------------------------------------
# Weather Anomaly (deviation from 30-day rolling seasonal baseline)
# A 10 C day in January is anomalously warm; the same 10 C in March is normal.
# High anomaly = unusual weather = bigger consumption spike.
# ---------------------------------------------------------
print("Calculating weather anomalies (30-day rolling baseline)...")
df_long['HDH_anomaly'] = (
    df_long['HDH'] - df_long.groupby('ClientID', observed=True)['HDH'].transform(
        lambda x: x.rolling(96 * 30, min_periods=96).mean()
    )
).astype(np.float32)

df_long['CDH_anomaly'] = (
    df_long['CDH'] - df_long.groupby('ClientID', observed=True)['CDH'].transform(
        lambda x: x.rolling(96 * 30, min_periods=96).mean()
    )
).astype(np.float32)

# ---------------------------------------------------------
# Drop NaN rows from lag/rolling operations
# ---------------------------------------------------------
print("Dropping NaNs...")
df_long = df_long.dropna().reset_index(drop=True)

print("Feature Engineering Complete!")
display(df_long)

## 8. Exporting the Processed Data
We save the result as a Parquet file. Parquet preserves our `float32` and `category` memory optimizations, drastically reducing both file size and load times for the modeling phase compared to a standard CSV.

In [8]:
display(df_long)

,Date,Weekday,Hour,Month,Is_Weekend,Is_Holiday,ClientID,Consumption,HDH,CDH,Lag_15min,Lag_1h,Lag_24h,Lag_1week,Rolling_Mean_4h
0,2012-01-08 00:15:00,7,1,2,True,False,MT_001,5.076142,8.141814,0.0,5.076142,19.035534,3.807106,3.807106,13.324873
1,2012-01-08 00:30:00,7,1,2,True,False,MT_001,5.076142,8.141814,0.0,5.076142,16.497461,3.807106,5.076142,12.531726
2,2012-01-08 00:45:00,7,1,2,True,False,MT_001,5.076142,8.141814,0.0,5.076142,5.076142,5.076142,3.807106,11.579949
3,2012-01-08 01:00:00,7,2,2,True,False,MT_001,5.076142,8.602323,0.0,5.076142,5.076142,3.807106,3.807106,10.786802
4,2012-01-08 01:15:00,7,2,2,True,False,MT_001,3.807106,8.602323,0.0,5.076142,5.076142,3.807106,5.076142,9.914340
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41548229,2014-12-31 23:00:00,3,24,13,False,False,MT_370,7621.621582,11.229535,0.0,7189.188965,7891.892090,18378.378906,7081.081055,8192.567383
41548230,2014-12-31 23:15:00,3,24,13,False,False,MT_370,6702.702637,11.229535,0.0,7621.621582,7945.945801,17351.351562,6864.864746,8040.540527
41548231,2014-12-31 23:30:00,3,24,13,False,False,MT_370,6864.864746,11.229535,0.0,6702.702637,7351.351562,18864.865234,6918.918945,7929.054199
41548232,2014-12-31 23:45:00,3,24,13,False,False,MT_370,6540.540527,11.229535,0.0,6864.864746,7189.188965,17621.621094,6594.594727,7804.054199


In [9]:
# This cell should take less than 30 seconds to run

print("Saving to Parquet format...")
df_long.to_parquet('../Datasets/processed_electricity_data.parquet', index=False)
print("Preprocessing complete! File saved as '../Datasets/processed_electricity_data.parquet'")

Saving to Parquet format...
Preprocessing complete! File saved as '../Datasets/processed_electricity_data.parquet'
